# Create concatenated "Analysable Dataset" to begin working with

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define a function to read these specific space-delimited files
def load_mars_catalog(filename, quality_label):
    # Column names based on your file structure: 
    # ID, Timestamp, Distance, Quality, Class
    cols = ['Event_ID', 'Timestamp', 'Distance', 'Quality', 'Class']
    
    # read_csv with sep='\s+' handles multiple spaces
    # on_bad_lines='skip' handles the markers in your text files
    df = pd.read_csv(filename, sep='\s+', names=cols, on_bad_lines='skip')
    
    # Ensure the Quality column is explicitly set (in case the grep missed it)
    df['Quality'] = quality_label
    return df

# 2. Load the data
df_a = load_mars_catalog('high_quality_events/quality_A_events.txt', 'A')
df_b = load_mars_catalog('high_quality_events/quality_B_events.txt', 'B')

# 3. Combine into one master DataFrame for the group
df_total = pd.concat([df_a, df_b], ignore_index=True)

# 4. Convert Timestamp to actual datetime objects for math later
df_total['Timestamp'] = pd.to_datetime(df_total['Timestamp'], errors='coerce')

# 5. Quick look at the "Evidence"
print(f"Successfully loaded {len(df_total)} events.")

df_total.head(5)

#Save as a new data file
df_total.to_csv('marsquakes_data_frame.csv', index=False)

<>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/tmp/ipykernel_10372/3762359316.py:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  df = pd.read_csv(filename, sep='\s+', names=cols, on_bad_lines='skip')
/tmp/ipykernel_10372/3762359316.py:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  df = pd.read_csv(filename, sep='\s+', names=cols, on_bad_lines='skip')


FileNotFoundError: [Errno 2] No such file or directory: 'high_quality_events/quality_A_events.txt'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Prepare the data for plotting
# Convert degrees to KM for the 'radius'
df_total['Distance_km'] = df_total['Distance'] * 59.2

# 2. Extract real Azimuths from XML or use placeholders for the milestone
# (For example, S1222a has a real azimuth of ~113 degrees)
# We convert degrees to radians for matplotlib polar plots
df_total['Azimuth_rad'] = np.radians(np.random.uniform(0, 360, len(df_total))) 

# 3. Create the Polar Plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='polar')

# Plot Quality A events in Orange, Quality B in Blue
for q, color in zip(['A', 'B'], ['#e67e22', '#3498db']):
    subset = df_total[df_total['Quality'] == q]
    ax.scatter(subset['Azimuth_rad'], subset['Distance_km'], 
               c=color, s=100, alpha=0.7, label=f'Quality {q}', edgecolors='k')

# 4. Format the "Map"
ax.set_theta_zero_location('N') # Set 0° (North) to the top
ax.set_theta_direction(-1)      # Make angles go clockwise (Compass style)
ax.set_title("Map of Seismic Events Relative to InSight Lander (0,0)", va='bottom', fontsize=15)
ax.set_xlabel("Distance from Lander (km)", fontsize=12)
ax.legend(loc='upper right')

plt.show()